# 054 — RNN, LSTM y secuencias

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** h₁ = tanh(1) = **0.7616**; h₂ = tanh(0.5·0.7616) = tanh(0.3808) =
**0.3634**; h₃ = tanh(0.1817) = **0.1797**. El estado decae aproximadamente a la
mitad por paso (≈ w_h): la RNN "olvida" el impulso geométricamente.

**Ejercicio 2.** |∂h_t/∂h_{t−k}| ≤ (|w_h|·max|tanh'|)^k ≤ 0.5^k.
k=5: 0.031; k=10: 9.8×10⁻⁴; k=20: 9.5×10⁻⁷. Desde **k = 10** la señal cae por debajo
de 10⁻³: dependencias más largas que ~10 pasos son casi inaprendibles aquí.

**Ejercicio 3.** c₁ = 0.9·2 + 0.5·1 = **2.3**; h₁ = tanh(2.3) = **0.9801**.
c₅ = 2.3·0.9⁴ = 2.3·0.6561 = **1.509** (66 % retenido). La RNN equivalente:
2.3·0.5⁴ = 0.144 (6 %). La compuerta de olvido cercana a 1 es memoria de largo plazo.

**Ejercicio 4.** LSTM: 4·(20·10 + 20·20 + 20) = 4·620 = **2480**. RNN simple: 1·620 =
**620**. La LSTM cuadruplica parámetros por sus tres compuertas más el candidato.


In [ ]:
result = run_lab("neural", seed=54)
assert result["kind"] == "neural"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica
import math

# Ejercicio 1
w_x, w_h, h = 1.0, 0.5, 0.0
hs = []
for x in [1.0, 0.0, 0.0]:
    h = math.tanh(w_x * x + w_h * h)
    hs.append(round(h, 4))
print("estados:", hs)
assert hs == [0.7616, 0.3634, 0.1797]

# Ejercicio 2
for k in (5, 10, 20):
    print(f"cota k={k}: {0.5**k:.2e}")

# Ejercicio 3
c1 = 0.9 * 2 + 0.5 * 1.0
h1 = 1.0 * math.tanh(c1)
c5 = c1 * 0.9 ** 4
print(f"c1={c1}  h1={h1:.4f}  c5={c5:.4f}  (RNN: {c1 * 0.5**4:.4f})")
assert abs(c1 - 2.3) < 1e-9

# Ejercicio 4
params_por_juego = 20 * 10 + 20 * 20 + 20
print("LSTM:", 4 * params_por_juego, "| RNN:", params_por_juego)


## Reflexión

1. ¿Por qué la actualización aditiva `c_t = f⊙c_{t−1} + i⊙c̃_t` preserva el gradiente donde `h_t = tanh(W_h·h_{t−1}+…)` lo destruye?
2. ¿Qué paralelismo hay entre la compuerta de olvido de la LSTM y el atajo residual de ResNet?
3. Para una serie temporal de sensores con 500 muestras y poco dato de entrenamiento, ¿qué argumentos darías a favor de una GRU frente a un Transformer?
